In [1]:
from pathlib import Path
import os
import subprocess
from reprojection_batch import run_reprojection_batch, copy_dlc_label_csvs_for_date
from prepare_calibration_dataset import extract_or_prepare_folder, restructure_and_downsample, downsample_dir
from video_utils import extract_frame, find_video_files

# Calibration Pipeline

End-to-end pipeline for calibrating a day of Rat Lockbox multi-camera
recordings using [Anipose](https://anipose.readthedocs.io/).

**Steps:**
1. **Prepare file structure**
   - Unzip a downloaded session archive and move it to the experiment directory.
   - Separate calibration videos (first timestamp group of 5 per box) from
     experiment videos and place them in `calibration/` and `videos-raw/`
     respectively.
   - Downsample calibration videos to the target FPS (default: 1 fps) to reduce
     Anipose calibration time.
2. **Run Anipose calibration** — invoke `anipose calibrate` in the session
   directory, which reads `config.toml` and writes `calibration.toml` for each
   box.  See `examples/anipose_config.toml` for a sample config.
3. **Extract frames for manual labeling** — pull a single frame from each
   camera's raw experiment video and add it to the DLC labeled-data folder so
   it can be manually labeled for reprojection validation.
4. **Batch reprojection test** — once manual labels exist, compute the mean
   reprojection error across all calibrations and label CSVs and save summary
   figures and a CSV report.

In [2]:
# Automatically create a list of all folder names in download_root directory
download_root = 'C:\\Users\\Gerardo\\Downloads'
folder_names  = [ #[f.stem for f in Path(zip_root).glob('*.zip')]
    # '03_08_2026-Day1_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task',
    # '04_08_2026_Day2_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task',
    # '05_08_2026_Day3_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task',
    # '06_08_2026_Day4_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task',
    # '07_08_2026_Day5_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task',
    # '10_08_2026_Day6_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task',
    '11_08_2026_Day7_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task',
    '12_08_2026_Day8_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task',
]
# folders to remove from folder_na<mes
exclude = []
# remove items in exclude from folder_names
folder_names = [name for name in folder_names if name not in exclude]

In [3]:
# Parameters for preprocessing (unzipping/prepping, moving, downsampling)
experiment     = 'sliding_lockbox'
target_dir     = f'C:\\Users\\Gerardo\\Documents\\Lockbox\\mlb2\\experiment\\' # Directory where the extracted and restructured videos will be saved
fps            = 3  # Desired frames per second for downsampled videos (e.g., 2 means 1 frame every 0.5 seconds)
unzip          = False   # Within the folder prepping step (moving from Downloads to experiment dir), skip unzipping
skip_moving_folder = True  # Set to True to skip the folder prepping step if already done

# Parameters for batch reprojection tests
dlc_model_path   = "C:\\Users\\Gerardo\\Documents\\Lockbox\\mlb2\\models\\calibration-rlb-2026-02-11"  # Path to trained DLC model with labeled data
calibration_path = "C:\\Users\\Gerardo\\Documents\\Lockbox\\mlb2\\video_calibration"  # Root directory with calibration scripts
labels_csv_root  = f"{calibration_path}\\manual_test_labels"  # Directory containing all manual label CSVs to use in batch
labels_date      = '2026-03-02'  # Date string to identify which DLC label CSVs to copy for batch reprojection

## 1. Prepare file structure
A. Unzip folder with a day's recordings. Assumes the following of folder organization:
- Root directory contains BOX2, BOX3, BOX4
- Each box folder contains 15 videos, the first 5 of which correspond to calibration videos from each camera
If assumptions are not met, may have to run parts of the pipeline below separately as needed

B. Separate calibration and experiment videos, and downsample calibration videos

In [10]:
for folder_name in folder_names:
    print(f"Preparing folder: {folder_name}")
    download_path = f'{download_root}\\{folder_name}'
    if unzip: download_path = f'{download_path}.zip'
    
    if not skip_moving_folder:
        extracted_folder = extract_or_prepare_folder(
            zip_path=download_path, target_dir=target_dir, unzip=unzip
        )
        print(f"Extracted to: {extracted_folder}")
    else:
        extracted_folder = Path(f"{target_dir}//{folder_name}")

    restructure_and_downsample(session_folder=extracted_folder, out_fps=fps, experiment=experiment)
    print(f"Preparation complete for: {folder_name}")

Preparing folder: 04_08_2026_Day2_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task
Calibration-target scan: 20260804_102836_camera06.mkv=0%, 20260804_102836_camera07.mkv=0%, 20260804_102836_camera08.mkv=0%, 20260804_102836_camera09.mkv=0%, 20260804_102836_camera10.mkv=0%
Calibration-target scan: 20260804_105846_camera06.mkv=0%, 20260804_105846_camera07.mkv=0%, 20260804_105846_camera08.mkv=0%, 20260804_105846_camera09.mkv=0%, 20260804_105846_camera10.mkv=0%
Calibration-target scan: 20260804_113619_camera06.mkv=0%, 20260804_113619_camera07.mkv=0%, 20260804_113619_camera08.mkv=0%, 20260804_113619_camera09.mkv=0%, 20260804_113619_camera10.mkv=0%
Calibration-target scan: 20260804_121039_camera06.mkv=67%, 20260804_121039_camera07.mkv=33%, 20260804_121039_camera08.mkv=58%, 20260804_121039_camera09.mkv=42%, 20260804_121039_camera10.mkv=0%
Created downsampled calibration video: C:\Users\Gerardo\Documents\Lockbox\mlb2\experiment\04_08_2026_Day2_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Tas

If need to downsample a specific folder of videos, e.g. because the order is not as assumed above, can use the code below

In [8]:
downsample_folders = {
    # '12_08_2026_Day8_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task': ['CAGE1'],
    # '11_08_2026_Day7_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task': ['CAGE1'],
    '07_08_2026_Day5_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task': ['CAGE1', 'CAGE2'],
    '06_08_2026_Day4_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task': ['CAGE1'],
    '05_08_2026_Day3_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task': ['CAGE1'],
    '04_08_2026_Day2_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task': ['CAGE1'],
    '03_08_2026-Day1_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task': ['CAGE1'],
}
for folder in downsample_folders: 
    for subfolder in downsample_folders[folder]:
        experiment_dir = Path(f"{target_dir}//{folder}//{subfolder}")
        downsample_dir(Path(f"{experiment_dir}//calibration"), out_fps=4, require_annotation=False) 

Created downsampled calibration video: C:\Users\Gerardo\Documents\Lockbox\mlb2\experiment\07_08_2026_Day5_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task\CAGE1\calibration\20260807_124203_camera06_4fps.avi
Created downsampled calibration video: C:\Users\Gerardo\Documents\Lockbox\mlb2\experiment\07_08_2026_Day5_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task\CAGE1\calibration\20260807_124203_camera07_4fps.avi
Created downsampled calibration video: C:\Users\Gerardo\Documents\Lockbox\mlb2\experiment\07_08_2026_Day5_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task\CAGE1\calibration\20260807_124203_camera08_4fps.avi
Created downsampled calibration video: C:\Users\Gerardo\Documents\Lockbox\mlb2\experiment\07_08_2026_Day5_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task\CAGE1\calibration\20260807_124203_camera09_4fps.avi
Created downsampled calibration video: C:\Users\Gerardo\Documents\Lockbox\mlb2\experiment\07_08_2026_Day5_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task\CAGE1\c

## 2. Calibrate videos with Anipose
Note: if a calibration fails or results in a large reprojection error, experimenting with the framerate can sometimes resolve this. 
If this occcurs, go to the cell above and run it to try downsampling to different framerates (sometimes higher or lower than the default of 2 helps).
1. Either delete or move all of the video and calibration files from the failed calibration to a new folder
2. Move the original videos from the "originals" folder to the parent "BOXx" directory
3. Run the cell above with the appropriate path

In [14]:
# Directory that contains config.toml (e.g., .../rat_lockbox/experiment)
config_path = Path(f"{target_dir}\\config.toml")

if not config_path.exists():
    raise FileNotFoundError(f"Missing config.toml at: {config_path}")

print(f"Running anipose calibrate in: {target_dir}")
result = subprocess.run(
    ["anipose", "calibrate"],
    cwd=target_dir,
    text=True,
    capture_output=True,
)

if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"anipose calibrate failed with code {result.returncode}")

print("Anipose calibration complete.")


Running anipose calibrate in: C:\Users\Gerardo\Documents\Lockbox\mlb2\experiment\
Calibrating...
C:\Users\Gerardo\Documents\Lockbox\mlb2\experiment\12_08_2026_Day8_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task\CAGE2
C:\Users\Gerardo\Documents\Lockbox\mlb2\experiment\12_08_2026_Day8_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task\CAGE2\calibration\calibration.toml
C:\Users\Gerardo\Documents\Lockbox\mlb2\experiment\12_08_2026_Day8_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task\CAGE1
C:\Users\Gerardo\Documents\Lockbox\mlb2\experiment\12_08_2026_Day8_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task\CAGE1\calibration\calibration.toml
no videos or calibration file found, continuing...
C:\Users\Gerardo\Documents\Lockbox\mlb2\experiment\11_08_2026_Day7_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task\CAGE2
C:\Users\Gerardo\Documents\Lockbox\mlb2\experiment\11_08_2026_Day7_Round2_Mouse_Lockbox2_Male_Cohort1_Pilot_10x10Task\CAGE2\calibration\calibration.toml
C:\Users\Gerardo\Docu

RuntimeError: anipose calibrate failed with code 1

## 3. Add frames and label manually
Run the code below to extract frames from the experiment videos in experiment_dir, and add to DLC labeled-data folder. Then must manually label points which are used to check reprojection error below

In [ ]:
subfolders = ['BOX2', 'BOX3', 'BOX4']

for folder_name in folder_names:
    experiment_dir = f'{target_dir}//{folder_name}'  # Directory with all experiment data
    video_files = find_video_files(experiment_dir, subfolders, ext="avi")
    label_paths = extract_frame(dlc_model_path, video_files, frame_index=100)

## 4. Test reprojections
Copies CSVs with labels and runs batches testing reprojection error on the manual labels

In [ ]:
copy_dlc_label_csvs_for_date(
    date_str=labels_date,
    dlc_model_path=dlc_model_path,
    target_folder=folder_name
)

In [ ]:
calibration_inputs = [
    # Session folder(s): automatically finds */calibration/calibration.toml
    rf"{target_dir}//240226_Rat_Lockbox_Cohort1_Combined_Lockbox_Task_Day1",
    rf"{target_dir}//250226_Rat_Lockbox_Males_Cohort1_Combined_Lockbox_Task_Day2",
    rf"{target_dir}//260226_Rat_Lockbox_Cohort1_Combined_Lockbox_Task_Day3",
]

# Images will be saved to: output_root / <session_name> / <box_name>
output_root = f"{calibration_path}\\reprojection_outputs"

for folder_name in folder_names:
    labels_csv_path = f"{labels_csv_root}\\{folder_name}"
    summary_df, batch_figures = run_reprojection_batch(
        calibration_inputs=calibration_inputs,
        label_csv_folder=labels_csv_path,
        output_root=output_root,
        image_size=(1920, 1080),
        max_charuco_frames_per_cam=1,
    )

summary_df.loc[:, ['calibration_box', 'calibration_day','mean_reprojection_error_px']].sort_values(by=['calibration_box','calibration_day'])